# Retraining from scratch

We run these models separately from the main experiment so that, when a new unlearned model is made, these checkpoints can be quickly pulled, measured, and then set aside again.

### Imports

In [1]:
import sys
print(sys.version)

3.9.25 (main, Apr 17 2026, 00:00:00) 
[GCC 11.5.0 20240719 (Red Hat 11.5.0-14)]


In [2]:
import os
import json

In [3]:
%ls

data/                              __pycache__/
evaluation/                        README.md
master_auditor.ipynb               results/
master_hyperparams.py              results_to_replicate.txt
master_pretraining.ipynb           trainer/
master_retrain_from_scratch.ipynb  unlearn/
master_unlearning.ipynb            visualize_pretraining_results.ipynb
models/                            visualize_results.ipynb
_old/                              wandb/


In [4]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
# import matplotlib.pyplot as plt

# from trainer.utils import training_regimen_lr_annealing


### Set configs for the pretraining

In [5]:

from master_hyperparams import hyperparams
device = "cuda" if torch.cuda.is_available() else "cpu"


# ------- MAIN THINGS TO EDIT FOR THIS RUN ------- #
dataset = "CIFAR10"
model_class = "ResNet"
unlearning_type = "random"
# ------------------------------------------------- #

hp = hyperparams[dataset]
model_hp = hp[model_class]

retrain_config = {

    "description": "Retrain from Scratch - Resnet, CIFAR10, random",
    
    "device": device,
    "model_class": model_class,
    "unlearning_type": unlearning_type,
    "data": {
        "dataset": dataset,
        "num_classes": hp["num_classes"],
        "batch_size": 1024,  # larger batch for faster pretraining
        "num_workers": hp["num_workers"],
        "item_to_unlearn": hp["items_to_unlearn"][unlearning_type]
        },

    "training": {
        "num_epochs": model_hp["training"]["num_epochs"],
        "num_runs": 3,
        "learning_rate": model_hp["training"]["learning_rate"],
        "weight_decay": model_hp["training"]["weight_decay"],
        "batch_print_freq": 12,
        },
}


### Protocol for several runs

In [6]:
import wandb
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /cs/student/msc/ml/2025/jmoncus/.netrc.
wandb: Currently logged in as: jjmoncus (jjmoncus706) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
import glob
from models.archs.utils import init_model
from torch.optim.lr_scheduler import ReduceLROnPlateau
from trainer.utils import training_regimen_lr_annealing
from data.dataloaders import load_dataloaders_for_experiment
import json
from data.utils import setup_seed
import time
from data.utils import split_forget_retain, split_random

def run_retrain_from_scratch(config, checkpoint_folder):
    
    print("="*70)
    print("="*19 + "  " + f'RUNNING RETRAINING FROM SCRATCH, SEED {config["GRAND_SEED"]}' + "  " + "="*19)
    print("="*70 + "\n")

    setup_seed(config["GRAND_SEED"])

    # create a subfolder for saving model checkpoints for this retraining
    print(f'All models will be of class {config["model_class"]}.\n')
    checkpoint_subfolder = os.path.join(checkpoint_folder, f"seed_{config['GRAND_SEED']}", "retrain_from_scratch")
    if not os.path.exists(checkpoint_subfolder):   
        print(f"{checkpoint_subfolder} doesn't exist - creating it...\n")
        os.makedirs(checkpoint_subfolder, exist_ok=True)


    # Save the config for this retraining to the main checkpoints folder
    with open(os.path.join(checkpoint_subfolder, "retrain_from_scratch_config.json"), "w") as f:
        json.dump(config, f, indent=4)

    # ... decide what we're unlearning
    item_to_unlearn = config["data"]["item_to_unlearn"]

    # ... announce what we're unlearning
    retrain_name = f"{config['data']['dataset']}_{config['model_class']}_{config['training']['num_epochs']}_epochs_{config['unlearning_type']}_{item_to_unlearn}"
    print("-"*15 + "    " + retrain_name + "\n")
    

    # ... be intelligent about setting `class_to_replace` or `percent_to_replace` if either is None
    
    # class_param = item_to_unlearn if config['unlearning_type'] == "class" else None
    # percent_param = item_to_unlearn if config['unlearning_type'] == "percent" else None

    # ----------------------------------------------------------------------------------- #
    # ----------------------------- DEFINE UNLEARNING LOADERS --------------------------- #
    # ----------------------------------------------------------------------------------- #

    # ...  ------------- get some unlearning data for this config ------------------- #
    # ... the dataSET is fixed across runs, and the randomness within runs is handled by simply shuffling the data loader. There is no need to actually apply the micro-seed

    # only need `train`
    marked_train_loader, _, _ = load_dataloaders_for_experiment(
        name = config["data"]["dataset"],
        replace_type=config["unlearning_type"], 
        value_to_replace=item_to_unlearn, 
        batch_size=config["data"]["batch_size"], 
        num_workers=config["data"]["num_workers"], 
        seed = config["GRAND_SEED"], 
        only_mark=True,
        val=False
        )
    # we make sure forget and retain sets are shuffled, to allow randomness across runs
    # only need `retain`
    print("Training - forget vs retain split:")
    _, retain_loader = split_forget_retain(marked_train_loader, batch_size=config["data"]["batch_size"], shuffle = True, num_workers=config["data"]["num_workers"])
    

    
    # ... and do a bunch of runs, where ...
    for i in range(1,  config["training"]["num_runs"]+1):

    
        # ----------------------------------------------------------------------------------- #
        # ----------------------------- RETRAIN FROM SCRATCH -------------------------------- #
        # ----------------------------------------------------------------------------------- #

        run_seed = config["GRAND_SEED"] * 1000 + i
        setup_seed(run_seed)
        
            # ... open new wandb session per run
        wandb.init(
            project="Verifying-Unlearning-2026",
            name=f"{run_seed}_retrain_{retrain_name}",
            config=config,
            reinit= "finish_previous"
            )
        
        print(f" ----- Retraining from scratch for run {i}, {retrain_name} ----- \n")
            
        # ... init a fresh model, opt, criterion, and scheduler for this run_seed
        empty_model = init_model(model_class = config["model_class"], num_classes = config['data']["num_classes"]).to(config["device"])
        opt = optim.Adam(empty_model.parameters(), lr=config["training"]["learning_rate"], weight_decay = config["training"]["weight_decay"])
        criterion = nn.CrossEntropyLoss()
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            opt, 
            T_max=config["training"]["num_epochs"], 
            eta_min=1e-6
            )

        # ... do the training
        retrain_name = f"retrain_run_{i}_{retrain_name}"
        retrain_checkpoint_path = os.path.join(checkpoint_subfolder, f"{retrain_name}.pth")
        start = time.time() # EVENTUALLY NEEDS TO BE MEASURED SOME OTHER WAY
        retrained_model, opt, scheduler, retrain_retain_loss, retrain_retain_acc, retrain_retain_entr, retrain_retain_m_entr = training_regimen_lr_annealing(
            empty_model, 
            retain_loader,
            opt, 
            criterion, 
            scheduler, 
            device = config["device"], 
            num_epochs=config["training"]["num_epochs"], 
            model_path = retrain_checkpoint_path,
            print_freq = config["training"]["batch_print_freq"],
            w_and_b = True
            )
        end = time.time()
        wandb.log({"run time efficiency": end - start})
        

        # closes retrain wandb session
        wandb.finish()


    print("-"*75)
    print("-"*19 + "  " + f'FINISHED RETRAIN FROM SCRATCH, SEED {config["GRAND_SEED"]}' + "  " + "-"*19)
    print("-"*75 + "\n")
    

/cs/student/project_msc/2025/ml/jmoncus/virtual-envs/vu2026/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Check metrics on unlearned models

In [8]:
# MAKE A RANDOM SEED
retrain_config["GRAND_SEED"] = 5
# DO EXP
run_retrain_from_scratch(config = retrain_config, checkpoint_folder="models/model_checkpoints")

===================  RUNNING RETRAINING FROM SCRATCH, SEED 5  ===================

setup random seed = 5
All models will be of class ResNet.

models/model_checkpoints/seed_5/retrain_from_scratch doesn't exist - creating it...

---------------    CIFAR10_ResNet_100_epochs_random_0.1

Replacing 5000 samples total (10.0%)
Replacing indeces: [23656 27442 40162  8459  8051 42404    89  1461 13519 42536] ...
========== DATALOADER INFO
Dataset: CIFAR-10
Train: 50000 images for training
Test: 10000 images for testing
Replace type = random, value to replace = 0.1
Training augmentation = randomcrop(32,4) + randomhorizontalflip + colorjitter + randomrotation + normalize
Validation/Test augmentation = normalize
num_workers = 4


Training - forget vs retain split:
Forget set: 5000 items
Retain set: 45000 items


setup random seed = 5001


 ----- Retraining from scratch for run 1, CIFAR10_ResNet_100_epochs_random_0.1 ----- 

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][11/44]	Loss 1.6723 (2.0517)	Accuracy 37.500 (26.074)	Entropy 1.6168 (1.8186)	M-Entropy 1.5991 (2.0062)	Time 3.79
Epoch: [1][23/44]	Loss 1.5931 (1.8426)	Accuracy 40.918 (32.585)	Entropy 1.5561 (1.7060)	M-Entropy 1.5221 (1.7842)	Time 2.52
Epoch: [1][35/44]	Loss 1.4552 (1.7233)	Accuracy 46.191 (36.602)	Entropy 1.4501 (1.6367)	M-Entropy 1.4000 (1.6621)	Time 2.53
train_accuracy (epoch) 38.796
Epoch 1 | LR: 1.0e-03 | RAM: 2.03GB | VRAM: 7.08GB | Weight Norm: 111.719
 ----- EPOCH 2 ----- 

Epoch: [2][11/44]	Loss 1.2692 (1.3112)	Accuracy 52.051 (51.969)	Entropy 1.2998 (1.3215)	M-Entropy 1.2359 (1.2727)	Time 3.30
Epoch: [2][23/44]	Loss 1.2723 (1.2724)	Accuracy 53.125 (53.499)	Entropy 1.2274 (1.2803)	M-Entropy 1.2712 (1.2412)	Time 2.53
Epoch: [2][35/44]	Loss 1.1406 (1.2400)	Accuracy 57.812 (54.891)	Entropy 1.1044 (1.2474)	M-Entro

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


RAM_GB,▁▃▆▆▇▇▇▇██████████████▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇
VRAM_GB,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇███
learning_rate,██████▇▇▇▇▇▇▇▆▆▅▅▅▅▄▄▄▄▄▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁
run time efficiency,▁
time (batch),▁▁▆▇▁▁▂▇▁▁▇▁▇▇▆▇▂▂█▂█▇▇▂▂▂██▁▆▁▁▆▆▂▁▁▂▂▇
train_acc (batch),▁▂▆▆▆▆▇▇▇▇▇▇▇█▇█████████████████████████
train_acc (full),▁▃▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇███████████████████████
train_entropy (batch),█▇▅▄▄▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_entropy (full),█▆▅▄▄▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


setup random seed = 5002


...

setup random seed = 5003


 ----- Retraining from scratch for run 3, retrain_run_2_retrain_run_1_CIFAR10_ResNet_100_epochs_random_0.1 ----- 

The normalize layer is contained in the network
 ----- EPOCH 1 ----- 

Epoch: [1][11/44]	Loss 1.7867 (2.0578)	Accuracy 34.766 (24.772)	Entropy 1.7338 (1.8214)	M-Entropy 1.6951 (2.0073)	Time 3.36
Epoch: [1][23/44]	Loss 1.5207 (1.8258)	Accuracy 44.727 (32.784)	Entropy 1.5189 (1.7109)	M-Entropy 1.4588 (1.7590)	Time 2.59
Epoch: [1][35/44]	Loss 1.4581 (1.7079)	Accuracy 45.020 (36.903)	Entropy 1.4384 (1.6328)	M-Entropy 1.4092 (1.6427)	Time 2.59
train_accuracy (epoch) 39.191
Epoch 1 | LR: 1.0e-03 | RAM: 2.10GB | VRAM: 7.09GB | Weight Norm: 111.676
 ----- EPOCH 2 ----- 

Epoch: [2][11/44]	Loss 1.2593 (1.3078)	Accuracy 54.688 (52.686)	Entropy 1.2554 (1.3396)	M-Entropy 1.2283 (1.2629)	Time 3.31
Epoch: [2][23/44]	Loss 1.2654 (1.2623)	Accuracy 52.832 (54.126)	Entropy 1.2095 (1.2759)	M-Entropy 1.2610 (1.2307)	Time 2.60
Epoch: [2][35/44]	Loss 1.1212 (1.2249)	Accuracy 60.645 (55.461)	Ent

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


RAM_GB,█████████████████████████▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
VRAM_GB,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
learning_rate,███████▇▇▇▇▇▇▇▆▆▆▆▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▂▁▁▁▁▁
run time efficiency,▁
time (batch),▇▇▆▆▇▂▂▂█▂▇▂▂▂██▁▁▇▁█▂▂█▂▂█▂▂▂▂▂▂▂█▂▂▂▂█
train_acc (batch),▁▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇▇▇██████████████████████
train_acc (full),▁▂▃▃▃▄▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇██████████████████
train_entropy (batch),█▆▅▄▄▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_entropy (full),█▇▅▅▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+5,...


---------------------------------------------------------------------------
-------------------  FINISHED RETRAIN FROM SCRATCH, SEED 5  -------------------
---------------------------------------------------------------------------

